# Amazon Bedrock AgentCore Runtime での Strands Agents を使用したストリーミング応答

## 概要

このチュートリアルでは、Amazon Bedrock AgentCore Runtime を使用してストリーミング応答を実装する方法を学習します。この例では、利用可能になったら部分結果をストリーミングする方法を示し、大量のコンテンツを生成する操作や処理に時間がかかる操作に対して、より応答性の高いユーザー体験を提供します。


### チュートリアルの詳細

|情報| 詳細|
|:--------------------|:---------------------------------------------------------------------------------|
| チュートリアルタイプ       | ストリーミング付き会話型|
| エージェントタイプ          | 単一         |
| エージェントフレームワーク   | Strands Agents |
| LLM モデル           | Anthropic Claude Haiku 4.5 |
| チュートリアルコンポーネント | AgentCore Runtime、Strands Agent、Amazon Bedrock Model を使用したストリーミング応答 |
| チュートリアル垂直領域   | クロス垂直                                                                   |
| 例の複雑さ  | 簡単                                                                             |
| 使用するSDK            | Amazon BedrockAgentCore Python SDK と boto3|

### チュートリアルアーキテクチャ

このチュートリアルでは、ストリーミングエージェントを AgentCore runtime にデプロイする方法について説明します。

デモンストレーションの目的で、ストリーミング機能を備えた Amazon Bedrock モデルを使用する Strands Agent を使用します。

この例では、`get_weather` と `get_time` の2つのツールを持つシンプルなエージェントを使用しますが、ストリーミング応答機能を備えています。

    
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>

### チュートリアルの主な機能

* Amazon Bedrock AgentCore Runtime でのエージェントからのストリーミング応答
* リアルタイム部分結果配信
* ストリーミング機能を備えた Amazon Bedrock モデルの使用
* 非同期ストリーミングサポートを備えた Strands Agents の使用

## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Python 3.10+
* AWS 認証情報
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker が実行中

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## AgentCore Runtime へのデプロイ用のストリーミングエージェントの準備

ストリーミングエージェントを AgentCore Runtime にデプロイしましょう。ストリーミング機能は、エントリーポイント関数で非同期ジェネレータまたは yield ステートメントを使用すると、AgentCore SDK によって自動的に処理されます。

ストリーミング実装の重要なポイント：
* エントリーポイント関数には `async def` を使用
* 利用可能になったら `yield` を使用してチャンクをストリーミング
* AgentCore SDK が自動的に Server-Sent Events (SSE) 形式を処理
* クライアントは Content-Type: text/event-stream 応答を受信

### Amazon Bedrock モデルとストリーミングを使用した Strands Agents
Amazon Bedrock モデルを使用した Strands Agent のストリーミング実装を見てみましょう。

In [1]:
%%writefile strands_claude_streaming.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel
import asyncio
from datetime import datetime

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"

@tool
def get_time():
    """ Get current time """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[
        calculator, weather, get_time
    ],
    system_prompt="""You're a helpful assistant. You can do simple math calculations, 
    tell the weather, and provide the current time."""
)

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    Invoke the agent with streaming capabilities
    This function demonstrates how to implement streaming responses
    with AgentCore Runtime using async generators
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            if "data" in event:
                yield event["data"]
            
    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response

if __name__ == "__main__":
    app.run()

Writing strands_claude_streaming.py


## AgentCore Runtime でのストリーミングの理解

AgentCore Runtime でストリーミングを使用すると、いくつかのことが自動的に行われます：

### Server-Sent Events (SSE) 形式
* AgentCore SDK が自動的に yield されたデータを SSE 形式に変換
* 各 yield が SSE ストリーム内の `data: ` イベントになる
* Content-Type が自動的に `text/event-stream` に設定される

### クライアント処理
* クライアントは、エージェントがリクエストを処理する際にリアルタイム更新を受信
* これにより、段階的な応答表示とより良いユーザー体験が可能
* クライアントは完全な応答が準備される前に部分結果を処理できる

### エラーハンドリング
* ストリーミング応答には適切なエラーハンドリングを含める必要がある
* エラーはストリームの一部として yield できる
* 関数が完了するか、未処理の例外が発生するとストリームが終了する

## ストリーミングエージェントを AgentCore Runtime にデプロイ

`CreateAgentRuntime` 操作は包括的な設定オプションをサポートし、コンテナイメージ、環境変数、暗号化設定を指定できます。プロトコル設定（HTTP、MCP）と認証メカニズムを設定して、クライアントがエージェントと通信する方法を制御することもできます。

**注意:** 運用のベストプラクティスは、コードをコンテナとしてパッケージ化し、CI/CD パイプラインと IaC を使用して ECR にプッシュすることです

このチュートリアルでは、Amazon Bedrock AgentCode Python SDK を使用してアーティファクトを簡単にパッケージ化し、AgentCore runtime にデプロイします。

### AgentCore Runtime デプロイの設定

次に、スターターツールキットを使用して、エントリーポイント、作成した実行ロール、requirements ファイルで AgentCore Runtime デプロイを設定します。また、スターターキットを設定して、起動時に Amazon ECR リポジトリを自動作成します。

設定ステップ中に、アプリケーションコードに基づいて Docker ファイルが生成されます

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [2]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name
region

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude_streaming.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_claude_streaming"
)
response

Entrypoint parsed: file=/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/strands_claude_streaming.py, bedrock_agentcore_name=strands_claude_streaming
Configuring BedrockAgentCore agent: strands_claude_streaming
Generated .dockerignore
Generated Dockerfile: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/Dockerfile
Generated .dockerignore: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/.dockerignore
Setting 'strands_claude_streaming' as default agent
Bedrock AgentCore configured: /Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/.bed

ConfigureResult(config_path=PosixPath('/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/Dockerfile'), dockerignore_path=PosixPath('/Users/oji/workspace/_docs/98_reInvent/workshop/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/01-streaming-agent-response/.dockerignore'), runtime='Docker', region='ap-northeast-1', account_id='195049633937', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### ストリーミングエージェントを AgentCore Runtime に起動

Docker ファイルができたので、ストリーミングエージェントを AgentCore Runtime に起動しましょう。これにより、Amazon ECR リポジトリと AgentCore Runtime が作成されます

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [3]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_claude_streaming' to account 195049633937 (ap-northeast-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_claude_streaming
✅ ECR repository available: 195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-strands_claude_streaming
Getting or creating execution role for agent: strands_claude_streaming
Using AWS region: ap-northeast-1, account ID: 195049633937
Role name: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-strands_claude_streaming


Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24
Starting execution role creation process for agent: strands_claude_streaming
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24
✓ Role created: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-strands_claude_streaming
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::195049633937:role/AmazonBedrockAgentCoreSDKRuntime-ap-northeast-1-2ce6205d24
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: strands_claude_streaming
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-northeast-1-2ce6205d24
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKC

### AgentCore Runtime のステータス確認
AgentCore Runtime をデプロイしたので、デプロイステータスを確認しましょう

In [4]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_claude_streaming


'READY'

### ストリーミングで AgentCore Runtime を呼び出し

最後に、ペイロードで AgentCore Runtime を呼び出し、ストリーミング応答を受信できます

<div style="text-align:left">
    <img src="images/invoke.png" width="85%"/>
</div>

In [5]:
invoke_response = agentcore_runtime.invoke({
    "prompt": 
    "天気はどうですか？"
})
invoke_response

天気情

報を確

認させ

ていただきます。

現

...

ご不

明な点がございましたら

、お気軽にお

尋ねください。

{}

### boto3 を使用したストリーミング用の AgentCore Runtime の呼び出し

AgentCore Runtime が作成されたので、任意の AWS SDK で呼び出すことができます。ストリーミング応答の場合、Server-Sent Events 形式を処理する必要があります。

In [6]:
import boto3
import json
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

# For streaming responses, we need to handle the EventStream
boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How much is 2+1"})
)

# Check if the response is streaming
if "text/event-stream" in boto3_response.get("contentType", ""):
    print("Processing streaming response with boto3:")
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                data = line[6:].replace('"', '')  # Remove "data: " prefix
                print(f"Received streaming chunk: {data}")
                content.append(data.replace('"', ''))
    
    # Display the complete streamed response
    full_response = " ".join(content)
    display(Markdown(full_response))
else:
    # Handle non-streaming response
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    
    if events:
        try:
            response_data = json.loads(events[0].decode("utf-8"))
            display(Markdown(response_data))
        except:
            print(f"Raw response: {events[0]}")

Processing streaming response with boto3:
Received streaming chunk: 2
Received streaming chunk:  
Received streaming chunk: + 1 = **
Received streaming chunk: 3**


2   + 1 = ** 3**

## ストリーミング応答の利点

ストリーミング応答には、いくつかの重要な利点があります：

### ユーザー体験
* **即座のフィードバック**: ユーザーは利用可能になったら部分結果を確認できる
* **パフォーマンスの向上感**: 合計時間が同じでも、応答がより速く感じられる
* **段階的表示**: 長い応答を段階的に表示できる

### 技術的利点
* **メモリ効率**: すべてをメモリに読み込むことなく、大きな応答を処理できる
* **タイムアウト防止**: 長時間実行される操作でのタイムアウトを回避
* **リアルタイム処理**: 利用可能になったらリアルタイムデータを処理できる

### ユースケース
* **コンテンツ生成**: 長文の執筆、レポート、ドキュメント
* **データ分析**: 複雑な計算からの段階的な結果
* **マルチステップワークフロー**: 複雑なエージェント推論を通じた進捗の表示
* **リアルタイム監視**: 監視エージェントからのライブ更新

## クリーンアップ（オプション）

作成した AgentCore Runtime をクリーンアップしましょう

In [7]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

('195049633937.dkr.ecr.ap-northeast-1.amazonaws.com/bedrock-agentcore-strands_claude_streaming',
 'strands_claude_streaming-LNGQJ5Fe3N',
 'bedrock-agentcore-strands_claude_streaming')

In [8]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# おめでとうございます！

Amazon Bedrock AgentCore Runtime を使用してストリーミングエージェントを正常に実装し、デプロイしました！

## 学習した内容：
* 非同期ジェネレータを使用したストリーミング応答の実装方法
* AgentCore Runtime が SSE 形式を自動的に処理する方法
* クライアント側でストリーミング応答を処理する方法
* ユーザー体験とパフォーマンスのためのストリーミングの利点

## 次のステップ：
* ユースケースに応じた異なるストリーミングパターンの実験
* 複雑なマルチステップワークフロー用のカスタムストリーミングロジックの実装
* Memory や Gateway などの他の AgentCore 機能とストリーミングを組み合わせる探索
* より良い UX のためのクライアント側ストリーミング可視化の実装を検討